# 🎓 EXP-0017: Teacher-Student Knowledge Recovery & Pose Fine-Tuning
### Nursing Home Fall Detection - Solusi Definitif untuk Upright Bias pada CCTV Miring

**Peneliti:** Raja Rahman Aziiz (`ml-modeling`)  
**Target Komputasi:** Kaggle T4 GPU / Google Colab / Local RTX GPU  
**Model Guru (Teacher):** `yolo11m-pose.pt` (Kapasitas Besar + TTA Rotasi 90°)  
**Model Murid (Student):** `yolo11n-pose.pt` (Ringan & Cepat untuk Edge Real-Time)  

---
### 📌 Latar Belakang & Hipotesis Riset:
1. **Akar Masalah (The Upright Bias):**
   Model pose standar COCO gagal mendeteksi orang terkapar mendatar di lantai pada kamera CCTV sudut plafon miring (misal kamera c2/c3 MCF), dengan *drop confidence* hingga < 15%.
2. **Solusi Sementara (TTA Rotasi):**
   Memutar gambar 90° berhasil memulihkan 100% deteksi pada evaluasi dataset, namun membebani FPS (turun 2-3x) dan rentan memicu *false positive* pada lingkungan keramaian banyak orang.
3. **Hipotesis EXP-0017 (Solusi Definitif):**
   Dengan menggunakan model guru (`yolo11m-pose` + TTA Rotasi) untuk melabeli ulang frame-frame rebah yang sebelumnya kosong di dataset latih, kita melatih ulang model murid (`yolo11n-pose`). Model murid hasil pelatihan baru ini diharapkan **mampu mendeteksi korban jatuh mendatar secara mandiri langsung pada gambar asli (NOL ROTASI, 30+ FPS, Zero Noise)!**


## 1. Setup Environment & Verifikasi Hardware GPU
Menginstal Ultralytics dan memeriksa ketersediaan akselerasi GPU (CUDA).


In [ ]:
# =============================================================================
# 1. SETUP ENVIRONMENT & VERIFIKASI HARDWARE GPU
# =============================================================================
import sys

# Cek apakah ultralytics sudah ada di environment Kaggle (hindari pip install yang lama jika sudah ada)
try:
    import ultralytics
    print(f"✅ Ultralytics sudah terpasang di sistem: v{ultralytics.__version__}")
except ImportError:
    print("📦 Menginstal ultralytics... Mohon tunggu...")
    # Dijalankan tanpa flag -q agar progress download & install terlihat jelas
    !pip install ultralytics
    import ultralytics
    print(f"✅ Berhasil menginstal Ultralytics v{ultralytics.__version__}")

import os
import math
import shutil
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import yaml
from PIL import Image
from tqdm.auto import tqdm

print("\n--- Verifikasi Hardware & GPU ---")
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Tersedia   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name     : {torch.cuda.get_device_name(0)}")
    device = "cuda:0"
else:
    print("⚠️ PERINGATAN: GPU tidak terdeteksi! Notebook berjalan pada CPU.")
    print("👉 Pastikan pada panel kanan Kaggle (Notebook Settings): Accelerator = GPU T4 x2 / P100")
    device = "cpu"


## 2. Deteksi & Persiapan Jalur Dataset (Kaggle / Local)
Mendeteksi apakah notebook berjalan di Kaggle environment (`/kaggle/input/`) atau lingkungan lokal.


In [ ]:
# =============================================================================
# 2. DETEKSI & PERSIAPAN JALUR DATASET (Kaggle / Local)
# =============================================================================
import os
import shutil
from pathlib import Path
from tqdm.auto import tqdm

kaggle_input = Path("/kaggle/input")
work_dir = Path("/kaggle/working") if kaggle_input.exists() else Path("ml/experiments/exp0017")
work_dir.mkdir(parents=True, exist_ok=True)

# 1. Cari dataset di input
dataset_path = None
if kaggle_input.exists():
    for p in sorted(kaggle_input.glob("**/images/val")):
        candidate = p.parent.parent
        if (candidate / "labels").exists():
            dataset_path = candidate
            break

if dataset_path is None:
    for p in [
        Path("/kaggle/input/fall-detection-dataset-urfall-mcf-3cls"),
        Path("/kaggle/input/datasets/lailansdct/fall-detection-dataset-urfall-mcf-3cls"),
        Path("ml/data/processed_final"),
        Path("../ml/data/processed_final")
    ]:
        if p.exists() and (p / "images" / "val").exists():
            dataset_path = p
            break

print(f"✅ Dataset Ditemukan: {dataset_path}", flush=True)
assert dataset_path is not None, "❌ Dataset tidak ditemukan! Pastikan dataset sudah di-attach di Kaggle."

data_work = work_dir / "dataset_exp0017"
print("🧹 Membersihkan folder kerja lama...", flush=True)
if os.name == "posix":
    os.system(f"rm -rf '{data_work}'")
else:
    shutil.rmtree(data_work, ignore_errors=True)

data_work.mkdir(parents=True, exist_ok=True)

# 2. Buat symlink gambar PER BERKAS dan siapkan label awal 56 kolom
# CATATAN KRUSIAL: Symlink harus per berkas (bukan folder) agar Ultralytics tidak
# me-resolve path kembali ke folder input yang read-only!
print("🔗 Menautkan gambar (symlink per berkas) & inisialisasi label 56 kolom...", flush=True)
for sp in ["train", "val", "test"]:
    img_dst_dir = data_work / "images" / sp
    lbl_dst_dir = data_work / "labels" / sp
    img_dst_dir.mkdir(parents=True, exist_ok=True)
    lbl_dst_dir.mkdir(parents=True, exist_ok=True)
    
    src_img_dir = dataset_path / "images" / sp
    src_lbl_dir = dataset_path / "labels" / sp
    
    imgs = sorted(list(src_img_dir.glob("*.jpg")) + list(src_img_dir.glob("*.png")))
    for img in imgs:
        # Symlink gambar per berkas
        dst_img = img_dst_dir / img.name
        if not dst_img.exists():
            try:
                dst_img.symlink_to(img.resolve())
            except OSError:
                shutil.copy2(img, dst_img)
        
        # Inisialisasi label awal 56 kolom (kelas 0: person + box + 17 keypoint v=0)
        src_lbl = src_lbl_dir / f"{img.stem}.txt"
        dst_lbl = lbl_dst_dir / f"{img.stem}.txt"
        
        lines_56 = []
        if src_lbl.exists():
            with open(src_lbl, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        # box asli (cx cy w h), kelas diset 0 (person)
                        box = parts[1:5]
                        # 17 keypoints padding v=0 (x=0.0, y=0.0, v=0)
                        kpts_padding = ["0.000000", "0.000000", "0"] * 17
                        row_56 = "0 " + " ".join(box) + " " + " ".join(kpts_padding)
                        lines_56.append(row_56)
        
        with open(dst_lbl, "w") as f:
            if lines_56:
                f.write("\n".join(lines_56) + "\n")

print("\n🔍 Verifikasi struktur dataset kerja:", flush=True)
for split in ["train", "val", "test"]:
    n_imgs = len(list((data_work / "images" / split).glob("*.*")))
    n_lbls = len(list((data_work / "labels" / split).glob("*.txt")))
    print(f"  • {split.upper():5s}: {n_imgs:4d} gambar | {n_lbls:4d} file label (semua 56 kolom valid!)", flush=True)

print(f"\n🎉 Dataset siap di: {data_work}", flush=True)


## 3. Modul Inferensi Guru (Teacher) dengan TTA Rotasi
Fungsi inferensi cerdas dengan rotasi 90° CW dan CCW untuk mengekstrak pose orang yang terkapar di lantai.


In [ ]:
ROTATIONS = {
    "cw": cv2.ROTATE_90_CLOCKWISE,
    "ccw": cv2.ROTATE_90_COUNTERCLOCKWISE
}

UNROTATE = {
    "cw": lambda x, y, w, h: (y, h - 1 - x),
    "ccw": lambda x, y, w, h: (w - 1 - y, x),
}

def predict_teacher_tta(model, img, imgsz=1280, conf=0.15, device=device):
    """
    Menjalankan model guru pada 3 sudut pandang (asli, CW 90, CCW 90).
    Mengembalikan deteksi dengan confidence tertinggi dalam format YOLO 17-keypoints.
    """
    h, w = img.shape[:2]
    all_dets = []

    # 1. Sudut pandang normal
    res = model(img, imgsz=imgsz, conf=conf, device=device, verbose=False)[0]
    if len(res.boxes) > 0:
        boxes = res.boxes.xyxy.cpu().numpy()
        confs = res.boxes.conf.cpu().numpy()
        kpts = res.keypoints.data.cpu().numpy()
        for bx, c, k in zip(boxes, confs, kpts):
            all_dets.append((float(c), bx, k, "normal"))

    # 2. Sudut pandang rotasi 90 CW dan CCW
    for name, code in ROTATIONS.items():
        rot_img = cv2.rotate(img, code)
        res_rot = model(rot_img, imgsz=imgsz, conf=conf, device=device, verbose=False)[0]
        if len(res_rot.boxes) == 0:
            continue
        boxes = res_rot.boxes.xyxy.cpu().numpy()
        confs = res_rot.boxes.conf.cpu().numpy()
        kpts = res_rot.keypoints.data.cpu().numpy()
        unrot = UNROTATE[name]
        for bx, c, k in zip(boxes, confs, kpts):
            corners = np.array([unrot(x, y, w, h) for x, y in ((bx[0], bx[1]), (bx[2], bx[3]))])
            box = np.array([corners[:, 0].min(), corners[:, 1].min(), corners[:, 0].max(), corners[:, 1].max()])
            k2 = k.copy()
            for i in range(17):
                k2[i, 0], k2[i, 1] = unrot(k[i, 0], k[i, 1], w, h)
            all_dets.append((float(c), box, k2, name))

    if not all_dets:
        return None

    # Ambil deteksi terbaik dengan confidence tertinggi
    best_det = max(all_dets, key=lambda d: d[0])
    return best_det


## 4. Tahap A: Pemulihan Pseudo-Label oleh Model Guru
Memindai dataset `train`, mencari frame yang kosong atau memiliki label tidak lengkap pada kasus rebahan, lalu memulihkan label menggunakan model guru `yolo11m-pose`.


In [ ]:
from ultralytics import YOLO

print("🚀 Memuat Model Guru: yolo11m-pose.pt...", flush=True)
teacher_model = YOLO("yolo11m-pose.pt")

train_img_dir = data_work / "images" / "train"
train_lbl_dir = data_work / "labels" / "train"

recovered_count = 0
pseudo_augmented_count = 0
audit_samples = []

train_images = sorted(list(train_img_dir.glob("*.jpg")) + list(train_img_dir.glob("*.png")))
print(f"Total gambar train: {len(train_images)}", flush=True)

pbar = tqdm(train_images, desc="Teacher Pseudo-Labeling", unit="img")
for img_path in pbar:
    lbl_path = train_lbl_dir / f"{img_path.stem}.txt"
    
    # Cek apakah label masih kosong / belum ada deteksi
    is_empty = False
    if not lbl_path.exists() or lbl_path.stat().st_size == 0:
        is_empty = True
    else:
        content = lbl_path.read_text().strip()
        if not content:
            is_empty = True

    img = None
    # JALUR 1: Jika label kosong (kasus orang rebahan yang luput dari anotasi asli)
    if is_empty:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]

        det = predict_teacher_tta(teacher_model, img, imgsz=1280, conf=0.15, device=device)
        if det is not None:
            conf, box, kpts, src_view = det
            x1, y1, x2, y2 = box
            cx = ((x1 + x2) / 2.0) / w
            cy = ((y1 + y2) / 2.0) / h
            bw = (x2 - x1) / w
            bh = (y2 - y1) / h

            kpts_norm = []
            for j in range(17):
                kx = kpts[j][0] / w
                ky = kpts[j][1] / h
                kv = 2 if kpts[j][2] > 0.25 else (1 if kpts[j][2] > 0.05 else 0)
                kpts_norm.extend([f"{kx:.6f}", f"{ky:.6f}", str(kv)])

            yolo_line = f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f} " + " ".join(kpts_norm) + "\n"
            with open(lbl_path, "w") as f:
                f.write(yolo_line)

            recovered_count += 1
            pbar.set_postfix({"Recovered": recovered_count, "View": src_view})
            if len(audit_samples) < 16:
                audit_samples.append((img_path, box, kpts, conf, src_view))
    
    # JALUR 2: Jika sudah punya box tapi keypoint masih nol (v=0)
    # Jalankan teacher untuk mendeteksi keypoint asli orang tersebut!
    else:
        # Periksa apakah keypoint masih nol
        lines = lbl_path.read_text().strip().splitlines()
        if lines and lines[0].endswith(" 0.000000 0.000000 0"):
            if img is None:
                img = cv2.imread(str(img_path))
            if img is not None:
                h, w = img.shape[:2]
                # Deteksi normal view dulu (cepat)
                res = teacher_model(img, imgsz=960, conf=0.20, device=device, verbose=False)[0]
                if len(res.boxes) > 0:
                    boxes = res.boxes.xyxy.cpu().numpy()
                    kpts = res.keypoints.data.cpu().numpy()
                    # Ambil deteksi terbesar / paling relevan
                    best_k = kpts[0]
                    best_b = boxes[0]
                    
                    x1, y1, x2, y2 = best_b
                    cx = ((x1 + x2) / 2.0) / w
                    cy = ((y1 + y2) / 2.0) / h
                    bw = (x2 - x1) / w
                    bh = (y2 - y1) / h
                    
                    kpts_norm = []
                    for j in range(17):
                        kx = best_k[j][0] / w
                        ky = best_k[j][1] / h
                        kv = 2 if best_k[j][2] > 0.25 else (1 if best_k[j][2] > 0.05 else 0)
                        kpts_norm.extend([f"{kx:.6f}", f"{ky:.6f}", str(kv)])
                    
                    yolo_line = f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f} " + " ".join(kpts_norm) + "\n"
                    with open(lbl_path, "w") as f:
                        f.write(yolo_line)
                    pseudo_augmented_count += 1

print(f"\n🎉 SELESAI PSEUDO-LABELING!")
print(f"  • Frame rebahan kosong yang dipulihkan (TTA Rotasi) : {recovered_count} frame")
print(f"  • Frame dengan pose keypoint diperkaya oleh guru   : {pseudo_augmented_count} frame")
print(f"  • SEMUA label train kini valid 56 kolom!")


## 5. Tahap B: Visual Audit Contact Sheet
Memvalidasi secara visual hasil pseudo-label guru untuk memastikan sendi pose menempel dengan presisi pada tubuh.


In [ ]:
import matplotlib.pyplot as plt

if audit_samples:
    n_samples = len(audit_samples)
    cols = 4
    rows = int(np.ceil(n_samples / cols))
    
    plt.figure(figsize=(18, 4.5 * rows))
    for idx, (img_path, box, kpts, conf, src_view) in enumerate(audit_samples):
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        for j in range(17):
            if kpts[j][2] > 0.20:
                cv2.circle(img_rgb, (int(kpts[j][0]), int(kpts[j][1])), 4, (255, 255, 0), -1)

        plt.subplot(rows, cols, idx + 1)
        plt.imshow(img_rgb)
        plt.title(f"{img_path.name}\nConf: {conf:.2f} ({src_view})", fontsize=9)
        plt.axis("off")
        
    plt.tight_layout()
    contact_sheet_path = work_dir / "exp0017_audit_contact_sheet.jpg"
    plt.savefig(contact_sheet_path, dpi=120)
    plt.show()
    print(f"✅ Contact sheet tersimpan di: {contact_sheet_path}")
else:
    print("Semua label train sudah terisi lengkap!")


## 6. Pembuatan File Konfigurasi `data_exp0017.yaml`
Menyiapkan path dataset untuk pelatihan YOLO-Pose.


In [ ]:
yaml_content = {
    "path": str(data_work.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "kpt_shape": [17, 3],
    "names": {0: "person"}
}

yaml_file = work_dir / "data_exp0017.yaml"
with open(yaml_file, "w") as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"✅ File data.yaml berhasil dibuat di: {yaml_file}")
print(yaml.dump(yaml_content))


## 7. Tahap C: Pelatihan Model Murid (`yolo11n-pose.pt`)
Melatih ulang model student `yolo11n-pose` pada dataset yang sudah diperkaya label guru.
Student kini belajar mengenali tubuh horizontal secara langsung tanpa memerlukan TTA rotasi!


In [ ]:
from ultralytics import YOLO

student_model = YOLO("yolo11n-pose.pt")

print("🚀 Memulai Pelatihan Student Model yolo11n-pose (EXP-0017)...")
results = student_model.train(
    data=str(yaml_file),
    epochs=100,
    imgsz=640,
    batch=16,
    device=device,
    patience=20,
    save=True,
    project=str(work_dir / "runs"),
    name="exp0017_student_yolo11n",
    exist_ok=True,
    pose=12.0,       # Bobot loss keypoints pose
    kobj=2.0,        # Bobot objectness keypoints
    box=7.5,
    cls=0.5,
    verbose=True
)

print("🎉 PELATIHAN MODEL STUDENT SELESAI!")


## 8. Evaluasi Mandiri pada Test Set Sulit (Zero-TTA Validation)
Menguji apakah model student baru mampu mendeteksi kasus rebahan sulit secara mandiri **TANPA ROTASI (Zero-TTA)**.


In [ ]:
best_weights = work_dir / "runs" / "exp0017_student_yolo11n" / "weights" / "best.pt"
if not best_weights.exists():
    best_weights = Path("yolo11n-pose.pt")

eval_model = YOLO(str(best_weights))
test_img_dir = data_work / "images" / "test"

hard_frames = sorted(list(test_img_dir.glob("mcf_s19_c2_*.jpg")))
print(f"Menguji {len(hard_frames)} frame mcf_s19_c2 secara mandiri (TANPA ROTASI)...\n")

detected_without_rot = 0
for f in tqdm(hard_frames, desc="Zero-TTA Hard Validation", unit="frame"):
    img = cv2.imread(str(f))
    res = eval_model(img, imgsz=640, conf=0.25, verbose=False)[0]
    if len(res.boxes) > 0:
        detected_without_rot += 1

rec_rate = (detected_without_rot / max(1, len(hard_frames))) * 100
print(f"\n✅ Hasil Deteksi Mandiri (Zero TTA): {detected_without_rot}/{len(hard_frames)} frame ({rec_rate:.1f}%)")
print("Bandingkan dengan baseline lama yang kehilangan 59.3% frame pada kamera c2!")


## 9. Simpan & Unduh Bobot Terbaik
Menyalin bobot terbaik `best.pt` ke root working folder untuk diunduh ke repositori lokal.


In [ ]:
export_path = work_dir / "exp0017_best.pt"
if best_weights.exists():
    shutil.copy(best_weights, export_path)
    print(f"✅ Model final siap diunduh: {export_path} (Ukuran: {export_path.stat().st_size / (1024*1024):.2f} MB)")
